# Lab 09 — CDN Edge Cache Simulation

The name of this lab project — **cdn-s3-lab** — promises a CDN. This lab delivers it.

A **Content Delivery Network** sits between clients and an origin object store.
Edge nodes hold frequently-accessed objects in memory (or on fast local disk).
A cache **hit** returns bytes instantly; a cache **miss** fetches from the origin
(RustFS in our case) and then caches the response for future requests.

### What you will build

```
Client
  │
  ├── CDNLayer
  │     ├── EdgeNode('us-east',  ttl=10s, latency=5ms)
  │     ├── EdgeNode('eu-west',  ttl=10s, latency=5ms)
  │     └── EdgeNode('ap-south', ttl=10s, latency=5ms)
  │           │  MISS → fetch from origin + cache
  │           └── RustFS (S3 origin, simulated 80ms latency)
```

Experiments:
1. Cache hit/miss ratio over repeated requests
2. TTL expiration and origin revalidation
3. Cache invalidation across edge nodes
4. Latency comparison: hit vs. miss
5. Hit ratio over time with different TTLs

---
## 1 · CDN Implementation

In [ ]:
from __future__ import annotations

import random
import sys
import time
from dataclasses import dataclass

sys.path.insert(0, '../scripts')
from lab_utils import cleanup_bucket, ensure_bucket, get_s3_client


@dataclass
class CachedObject:
    data: bytes
    cached_at: float
    ttl: float

    @property
    def expired(self) -> bool:
        return time.monotonic() - self.cached_at > self.ttl


class EdgeNode:
    """An in-memory CDN edge node with TTL-based expiry and simulated network latency."""

    def __init__(self, name: str, ttl: float = 10.0, latency_ms: float = 5.0):
        self.name = name
        self.ttl = ttl
        self.latency_ms = latency_ms  # simulated round-trip to return cached data
        self._cache: dict[str, CachedObject] = {}
        self.hits = 0
        self.misses = 0

    def get(self, key: str) -> bytes | None:
        """Return cached bytes or None on miss/expiry."""
        entry = self._cache.get(key)
        if entry is None or entry.expired:
            if entry:
                del self._cache[key]  # evict expired
            self.misses += 1
            return None
        self.hits += 1
        time.sleep(self.latency_ms / 1000)  # simulate edge latency
        return entry.data

    def put(self, key: str, data: bytes) -> None:
        self._cache[key] = CachedObject(data=data, cached_at=time.monotonic(), ttl=self.ttl)

    def invalidate(self, key: str) -> None:
        self._cache.pop(key, None)

    def flush(self) -> None:
        self._cache.clear()
        self.hits = 0
        self.misses = 0

    @property
    def hit_ratio(self) -> float:
        total = self.hits + self.misses
        return self.hits / total if total else 0.0

    @property
    def cached_keys(self) -> list[str]:
        return [k for k, v in self._cache.items() if not v.expired]


class CDNLayer:
    """Multi-edge CDN backed by an S3 origin (RustFS)."""

    ORIGIN_LATENCY_MS = 80  # simulated S3 round-trip latency

    def __init__(self, edges: list[EdgeNode], s3_client, bucket: str):
        self.edges = {e.name: e for e in edges}
        self._s3 = s3_client
        self._bucket = bucket

    def request(self, edge_name: str, key: str) -> tuple[bytes, str]:
        """Serve a request from the named edge. Returns (data, 'HIT'|'MISS')."""
        edge = self.edges[edge_name]
        data = edge.get(key)
        if data is not None:
            return data, 'HIT'
        # Cache miss — fetch from origin
        time.sleep(self.ORIGIN_LATENCY_MS / 1000)  # simulate origin latency
        try:
            data = self._s3.get_object(Bucket=self._bucket, Key=key)['Body'].read()
        except self._s3.exceptions.NoSuchKey:
            raise KeyError(f'Object not found in origin: {key}')
        edge.put(key, data)
        return data, 'MISS'

    def invalidate_all(self, key: str) -> None:
        """Propagate invalidation to all edges."""
        for edge in self.edges.values():
            edge.invalidate(key)

    def stats(self) -> dict:
        return {
            name: {'hits': e.hits, 'misses': e.misses, 'hit_ratio': e.hit_ratio}
            for name, e in self.edges.items()
        }


print('✅ CDN classes ready')

---
## 2 · Load Test Data into RustFS (Origin)

In [ ]:
s3 = get_s3_client()
BUCKET = 'lab9-cdn'
ensure_bucket(s3, BUCKET)

ASSETS = {
    'images/hero.jpg':      b'JPEG_BINARY_DATA_' + bytes(range(200)),
    'images/logo.png':      b'PNG_BINARY_DATA_'  + bytes(range(50)),
    'videos/intro.mp4':     b'MP4_BINARY_DATA_'  + bytes(range(500)),
    'css/styles.css':       b'body { margin: 0; } /* v2 */',
    'js/app.js':            b'console.log("hello");',
}

for key, data in ASSETS.items():
    s3.put_object(Bucket=BUCKET, Key=key, Body=data)

print(f'✅ Loaded {len(ASSETS)} assets into origin bucket {BUCKET!r}')
for key in ASSETS:
    print(f'   s3://{BUCKET}/{key}')

---
## Experiment 1 — Hit/Miss Ratio with Repeated Requests

In [ ]:
edges = [
    EdgeNode('us-east',  ttl=30, latency_ms=5),
    EdgeNode('eu-west',  ttl=30, latency_ms=5),
    EdgeNode('ap-south', ttl=30, latency_ms=5),
]
cdn = CDNLayer(edges, s3, BUCKET)

REQUESTS = 30
keys = list(ASSETS.keys())

print(f'Simulating {REQUESTS} requests to us-east (Zipf-like distribution)...')
print(f'{"#":>3}  {"Key":<25} {"Result":<6} {"Hit ratio"}')
print('-' * 50)

# Zipf-like: hero.jpg is 60% of traffic, others share the rest
weights = [0.60, 0.15, 0.10, 0.10, 0.05]

for i in range(1, REQUESTS + 1):
    key = random.choices(keys, weights=weights)[0]
    data, outcome = cdn.request('us-east', key)
    edge = cdn.edges['us-east']
    print(f'{i:>3}  {key:<25} {outcome:<6} {edge.hit_ratio:.0%}')

print()
print(f'Final hit ratio for us-east: {cdn.edges["us-east"].hit_ratio:.1%}')

---
## Experiment 2 — TTL Expiration and Revalidation

In [ ]:
# Use a very short TTL to observe expiry within the notebook
short_ttl_edge = EdgeNode('us-east-short', ttl=2.0, latency_ms=5)
short_cdn = CDNLayer([short_ttl_edge], s3, BUCKET)

KEY = 'css/styles.css'

_, outcome = short_cdn.request('us-east-short', KEY)
print(f'Request 1: {outcome}  (expected MISS — cold cache)')

_, outcome = short_cdn.request('us-east-short', KEY)
print(f'Request 2: {outcome}  (expected HIT  — just cached)')

print('Waiting 3 seconds for TTL to expire...')
time.sleep(3)

_, outcome = short_cdn.request('us-east-short', KEY)
print(f'Request 3: {outcome}  (expected MISS — TTL expired, re-fetched from origin)')

_, outcome = short_cdn.request('us-east-short', KEY)
print(f'Request 4: {outcome}  (expected HIT  — freshly cached again)')

---
## Experiment 3 — Cache Invalidation Across Edges

In [ ]:
KEY = 'js/app.js'

# Warm all edges
for edge_name in ['us-east', 'eu-west', 'ap-south']:
    cdn.request(edge_name, KEY)

print('All edges warmed — js/app.js is cached everywhere')
for name, edge in cdn.edges.items():
    print(f'  {name}: cached_keys={edge.cached_keys}')

# Simulate a deploy: update the file in origin and invalidate CDN
NEW_CONTENT = b'console.log("hello v2");  // deployed 2024'
s3.put_object(Bucket=BUCKET, Key=KEY, Body=NEW_CONTENT)
cdn.invalidate_all(KEY)
print(f'\nDeployed new version of {KEY} — invalidated all edges')

# Next request to any edge should be a MISS (re-fetches new version)
data, outcome = cdn.request('eu-west', KEY)
print(f'eu-west after invalidation: {outcome} → content={data!r}')
assert data == NEW_CONTENT, 'Should have fetched the new version!'
print('✅ eu-west now serves the updated file')

---
## Experiment 4 — Latency: HIT vs. MISS

In [ ]:
import matplotlib.pyplot as plt

test_edge = EdgeNode('latency-test', ttl=60, latency_ms=5)
test_cdn = CDNLayer([test_edge], s3, BUCKET)
KEY = 'images/hero.jpg'
RUNS = 10

miss_times, hit_times = [], []

# First request: MISS (fetches from origin)
t0 = time.perf_counter()
test_cdn.request('latency-test', KEY)
miss_times.append((time.perf_counter() - t0) * 1000)

# Subsequent requests: HIT
for _ in range(RUNS - 1):
    t0 = time.perf_counter()
    test_cdn.request('latency-test', KEY)
    hit_times.append((time.perf_counter() - t0) * 1000)

avg_miss = sum(miss_times) / len(miss_times)
avg_hit  = sum(hit_times)  / len(hit_times)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['MISS (origin fetch)', 'HIT (edge cache)'], [avg_miss, avg_hit],
       color=['#e74c3c', '#2ecc71'])
ax.set_ylabel('Latency (ms)')
ax.set_title(f'CDN Latency: HIT vs. MISS\n'
             f'(origin={CDNLayer.ORIGIN_LATENCY_MS}ms simulated, edge={test_edge.latency_ms}ms)')
for i, (label, val) in enumerate([('MISS', avg_miss), ('HIT', avg_hit)]):
    ax.text(i, val + 0.5, f'{val:.1f} ms', ha='center', fontweight='bold')

speedup = avg_miss / avg_hit if avg_hit > 0 else float('inf')
ax.text(0.5, 0.95, f'Cache HIT is {speedup:.1f}× faster',
        transform=ax.transAxes, ha='center', va='top', fontsize=11,
        color='#27ae60', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Experiment 5 — Hit Ratio Over Time with Different TTLs

In [ ]:

TTL_VALUES = [5, 15, 60]   # seconds — in our simulation we don't actually wait; we fake time
TOTAL_REQUESTS = 50
INTER_REQUEST_GAP = 0.5    # simulated seconds between requests

fig, ax = plt.subplots(figsize=(10, 5))

for ttl in TTL_VALUES:
    # Replay requests with synthetic timestamps
    # We monkey-patch time.monotonic so we can simulate time passing without sleeping
    cache: dict[str, tuple[float, bytes]] = {}  # key -> (cached_at, data)
    hits, misses = 0, 0
    ratios = []
    sim_time = 0.0

    for req_num in range(TOTAL_REQUESTS):
        key = random.choices(list(ASSETS.keys()), weights=[0.60, 0.15, 0.10, 0.10, 0.05])[0]
        entry = cache.get(key)
        if entry and (sim_time - entry[0]) < ttl:
            hits += 1
        else:
            misses += 1
            cache[key] = (sim_time, ASSETS[key])
        total = hits + misses
        ratios.append(hits / total)
        sim_time += INTER_REQUEST_GAP

    ax.plot(range(1, TOTAL_REQUESTS + 1), [r * 100 for r in ratios],
            label=f'TTL={ttl}s', linewidth=2)

ax.set_xlabel('Request number')
ax.set_ylabel('Cumulative hit ratio (%)')
ax.set_title('CDN Hit Ratio vs. TTL (Zipf-like traffic, 5 assets)')
ax.axhline(80, color='gray', linestyle='--', alpha=0.5, label='80% target')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Observation: higher TTL → higher hit ratio, but stale content risk increases.')
print('Use cache invalidation (Experiment 3) to manage staleness without sacrificing hit ratio.')

---
## Cleanup

In [ ]:
cleanup_bucket(s3, BUCKET)
print('✅ Cleanup complete')

---
## Summary

You built a working CDN simulation on top of RustFS:

| Component | Responsibility |
|-----------|---------------|
| `EdgeNode` | In-memory cache with TTL expiry and simulated latency |
| `CDNLayer` | Routes requests to the right edge, falls back to S3 origin on miss |
| Invalidation | Propagates content updates to all edges instantly |

### Key insights

- **Zipf's law matters**: 20% of assets serve 80% of requests → even a small cache with the right TTL achieves high hit ratios
- **TTL trade-off**: longer TTL = higher hit ratio but potentially stale content; solve with explicit invalidation on deploy
- **Cache miss cost**: 80ms origin fetch vs. 5ms edge hit = 16× latency difference

This is exactly what AWS CloudFront, Cloudflare, and Fastly do — at planet scale.
